In [2]:
# Make npz (Morgan fingerprint + RDKit descriptors)

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, Descriptors
from sklearn.preprocessing import StandardScaler

n_bits = 2048
radius = 2

out_y_csv = "properties.csv"
out_npz = f"feature_list_mfp{n_bits}_rdd.npz"

df = pd.read_csv("../data/zinc_dft/zinc_dft.csv")

# remove negative values and NaN
df["absorption wavelength"] = pd.to_numeric(df["absorption wavelength"], errors="coerce")
df["intensity"] = pd.to_numeric(df["intensity"], errors="coerce")
df = df.dropna(subset=["smiles", "absorption wavelength", "intensity"])
df = df[(df["absorption wavelength"] >= 0) & (df["intensity"] >= 0)].copy()

DESC_LIST = Descriptors._descList
mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

fps = []
descs = []
keep_rows = []

for i, row in df.iterrows():
    smi = str(row["smiles"])
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue

    # fingerprint
    fp = mfgen.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    # descriptors
    d = []
    ok = True
    for _, fn in DESC_LIST:
        try:
            v = float(fn(mol))
        except Exception:
            ok = False
            break
        if not np.isfinite(v):
            ok = False
            break
        d.append(v)

    if not ok:
        continue

    fps.append(arr)
    descs.append(d)
    keep_rows.append(i)

df_ok = df.loc[keep_rows].copy().reset_index(drop=True)

X_fp = np.vstack(fps).astype(np.int8)
X_desc = np.asarray(descs, dtype=np.float64)

# normalize rdkit descriptors
scaler = StandardScaler()
X_desc_z = scaler.fit_transform(X_desc).astype(np.float32)  # float32

# y
y = df_ok[["absorption wavelength", "intensity"]]
y.to_csv(out_y_csv, index=False)

# save npz
np.savez_compressed(out_npz, features_1=X_fp, features_2=X_desc_z)

print(f"Rows kept: {len(df_ok)}")
print(f"Saved y: {out_y_csv}")
print(f"Saved npz: {out_npz}")
print(f"X_fp: {X_fp.shape} dtype={X_fp.dtype}")
print(f"X_desc_z: {X_desc_z.shape} dtype={X_desc_z.dtype}")

Rows kept: 87628
Saved y: properties.csv
Saved npz: feature_list_mfp2048_rdd.npz
X_fp: (87628, 2048) dtype=int8
X_desc_z: (87628, 217) dtype=float32


In [ ]:
# Make csv (Morgan fingerprint)

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

input_csv_path = "input.csv"

n_bits = 1048
radius = 2

out_y_csv = "properties.csv"
out_x_csv = f"feature_list_mfp{n_bits}.csv"

df = pd.read_csv("../data/zinc_dft/zinc_dft.csv")

# remove negative values and NaN
df["absorption wavelength"] = pd.to_numeric(df["absorption wavelength"], errors="coerce")
df["intensity"] = pd.to_numeric(df["intensity"], errors="coerce")
df = df.dropna(subset=["smiles", "absorption wavelength", "intensity"])
df = df[(df["absorption wavelength"] >= 0) & (df["intensity"] >= 0)].copy()

fps = []
keep_rows = [] # to remove failed rows

mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

for i, row in df.iterrows():
    smi = str(row["smiles"])
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue

    fp = mfgen.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    fps.append(arr)
    keep_rows.append(i)

df_ok = df.loc[keep_rows].copy().reset_index(drop=True)
X = np.vstack(fps).astype(np.int8)

y = df_ok[["absorption wavelength", "intensity"]]
y.to_csv(out_y_csv, index=False)

np.savetxt(out_x_csv, X, fmt="%d", delimiter=",") # too large to commit (350MB when 2048)

print(f"Rows kept: {len(df_ok)}")